# Get Map layers and Update CMS  
This is not finished yet. Because there are so many changes, it is much simpler to just flush out the whole thing. We'll need to update at some point. 

## Set up

In [47]:
from arcgis.gis import GIS
import pandas as pd
from dotenv import load_dotenv
import os
import json

load_dotenv()

signin_name = os.getenv('ESRI_USERNAME')
signin_password = os.getenv('ESRI_PASSWORD')
signin_link = os.getenv("HUBLINK")

# Connect to the organization
gis = GIS(signin_link, signin_name, signin_password )
username = gis.users.me.username

print( "username" , username )
print("Done")

username sara2263_thrive_geohub
Done


In [48]:
maps = {
	'natural-treasures': {
		'title': 'Natural Treasures',
		'mapId': 'aba702c5420e4a36ac645f14a00ba8f1'
	},
	'transportation-infrastructure': {
		'title': 'Transportation + Infrastructure',
		'mapId': '25c75ac490b94b85b02aa4fd2f341fbb'
	},
	'regional-activity': {
		'title': 'Thrive Projects Activity Map Survey',
		'mapId': '79e19dd5e63645beb992122c502b69e5'
	}
}

## Get CMS Layer first to See Formatting  
Data is backed-up to Dropbox with todays date, in **TSV format**. The CMS content has a lot of commas

In [49]:
from arcgis.features import Table

CMS_URL = "https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/rest/services/CMS_DataDetails/FeatureServer/0"

cms = Table(CMS_URL)
cms_data = cms.query(as_df=True)
cms_data['sublayer_id'] = cms_data['sublayer_id'].apply( lambda x: str(x).split('.')[0] if not pd.isna(x) else None )
cms_data.tail()

,map_slug,map_title,map_id,group_path,node_title,node_type,item_id,kind,sublayer_id,url,summary,description,new_description,source,cross_link,FID,GlobalID,data_source_link,inter_hub_link_name,inter_hub_link_url
104,transportation-infrastructure,Transportation + Infrastructure,25c75ac490b94b85b02aa4fd2f341fbb,Thrive boundaries and mask layers,thrive_secondary_boundary,ArcGISFeatureLayer,562632e4f4c447ff9f1accd4a7640bb6,layer,NaN,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...,<NA>,<NA>,<NA>,<NA>,<NA>,105,17a67815-6027-489b-b8b3-2803b887c088,<NA>,<NA>,<NA>
105,transportation-infrastructure,Transportation + Infrastructure,25c75ac490b94b85b02aa4fd2f341fbb,Thrive boundaries and mask layers,Thrive Counties,GeoJSON,f97879f1e2b3463597ad7a0e43248d92,layer,NaN,https://www.arcgis.com/sharing/rest/content/it...,<NA>,<NA>,<NA>,<NA>,<NA>,106,a4ef8820-dd7e-4feb-99ed-3b5491d9c450,<NA>,<NA>,<NA>
106,<NA>,<NA>,25c75ac490b94b85b02aa4fd2f341fbb,<NA>,Where do People Live and Work,<NA>,6558c3383fa0413fb4b34b48f7e10b27,layer,NaN,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...,<NA>,<NA>,Commuting network lines showing home to work c...,U.S. Census Bureau,<NA>,107,2ab776b5-5f96-4f9b-8c27-4c8e6ec6dd37,<NA>,<NA>,<NA>
107,<NA>,<NA>,25c75ac490b94b85b02aa4fd2f341fbb,<NA>,Freight Infrastructure,<NA>,70ac6b0171e14336a7c0a17ef74c3e06,group,NaN,<NA>,<NA>,<NA>,"Where are freight bottlenecks, and what invest...",<NA>,<NA>,108,5310f2e2-e613-4137-85c2-2585dc5c3a0e,<NA>,<NA>,<NA>
108,<NA>,<NA>,25c75ac490b94b85b02aa4fd2f341fbb,<NA>,Commodity Flow Origin + Destination Network,<NA>,016928e7640f47b3b603f5212aaae210,<NA>,NaN,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...,<NA>,<NA>,Freight shipments between the region and other...,Derived from 2022 CFS Public Use Micro data Sa...,<NA>,109,732303ec-b541-4c18-8c0e-3ac9027fdbbb,<NA>,<NA>,<NA>


In [50]:
filename = os.path.join(
    r"/mnt/c/Users/cansu/Dropbox/studio2263/Thrive/data/cms-backup",
    f"{pd.Timestamp.now().strftime("%Y%m%d")}_Thrive_CMSArchieve.csv"
)
cms_data.to_csv(filename, index=False, sep="\t")

filename = os.path.join(
    r"/mnt/c/Users/cansu/Dropbox/studio2263/Thrive/data/cms-backup",
    f"{pd.Timestamp.now().strftime("%Y%m%d")}_Thrive_CMSArchieve.json"
)
cms_data.to_json(filename )

## Per Map  


In [51]:
# one row per node: the group itself, then every layer inside it
import urllib.request

rows = []
service_layers = {}

for slug, m in maps.items():
	wm = gis.content.get(m['mapId']).get_data()

	for top in wm.get('operationalLayers', []):
		if 'MASK_' not in top['title']:
			is_group = top['layerType'] == 'GroupLayer'

			# the group layer itself is a row.
			if is_group:
				try:
					top_item = top['itemId']
				except:
					top_item = None
					print( f"***Group Layer {top['title']} is not saved****" )

                # Record Group only props here. 
				rows.append({'map_slug': slug,
				             'map_title': m['title'],
				             'map_id': m['mapId'],
				             'group': top['title'],
				             'layer_title': None,
				             'layer_type': top['layerType'],
				             'kind': 'group',
				             'item_id': top_item,
				             'sublayer_id': None,
				             'url': None})

			# Layers are here. -- If there are groups nested in Groups this will crap out.
			for lyr in top['layers']:
				if ('MASK_' not in lyr['title']) and ('Tennessee Tax Parcels' not in lyr['title']) and ('Tax Parcel' not in lyr['title']):
					url = lyr['url']
					tail = url.rstrip('/').split('/')[-1]

					level = 'layer'
					sublayer_id = None
					
					# Deciding if the layer is a sublayer
					if tail.isdigit():

						# a service holding a single layer is that layer, not a sublayer of itself
						root = url.rstrip('/').rsplit('/', 1)[0]
						if root not in service_layers:
							info = json.loads(urllib.request.urlopen(root + '?f=json').read())
							service_layers[root] = len(info.get('layers') or []) + len(info.get('tables') or [])
						if service_layers[root] > 1:
							level = 'sublayer'
							sublayer_id = str(int(tail))

					rows.append({'map_slug': slug,
					             'map_title': m['title'],
					             'map_id': m['mapId'],
					             'group': top['title'] if is_group else '',
					             'layer_title': lyr['title'],
					            #  'layer_type': lyr['layerType'],
					             'item_id': lyr['itemId'],
					             'kind': level,
					             'sublayer_id': sublayer_id,
					             'url': url})


df = pd.DataFrame(rows)

df.head()


***Group Layer Thrive Region Masks is not saved****


,map_slug,map_title,map_id,group,layer_title,layer_type,kind,item_id,sublayer_id,url
0,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Cross Sector Data,NaN,GroupLayer,group,b12549616dcc4ff9bc75227052f120dd,NaN,NaN
1,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Cross Sector Data,Flood Hazard Areas,NaN,layer,9222394b0bf84b3f80a1da78c6c96580,NaN,https://services5.arcgis.com/7weheFjxuNkGGiZi/...
2,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Cross Sector Data,Population Change 2000-2020,NaN,sublayer,5239d5d3ea5f4d13a325ad0184377963,34,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...
3,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Cross Sector Data,Cultural + Historical Sites,NaN,layer,8fab6b2091f4410c8a241962457937db,NaN,https://services.arcgis.com/UnTXoPXBYERF0OH6/a...
4,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Cover,NaN,GroupLayer,group,d4f639dd817a435fb4bb68fc6102b9c9,NaN,NaN


In [52]:
print( df.shape[0], "data items retrieved from all maps" )

print( "\nData items per map:" )
df['map_id'].value_counts() 

102 data items retrieved from all maps

Data items per map:


map_id
25c75ac490b94b85b02aa4fd2f341fbb    50
aba702c5420e4a36ac645f14a00ba8f1    33
79e19dd5e63645beb992122c502b69e5    19
Name: count, dtype: int64

### There are 3 major data types here. Group, Layer, sublayer

In [53]:
# # Esri <NA> becomes "<NA> in some cases. 
# cms_data['sublayer_id'] = cms_data['sublayer_id'].replace( "<NA>", None )
# cms_data['sublayer_id'] = cms_data['sublayer_id'].apply( lambda x: str(int(x)) if not pd.isna(x) else None )

In [54]:
pd.set_option( "display.max_columns", None )
keys = ['map_id','item_id','sublayer_id', 'kind']

both = cms_data.merge(
    df[keys].drop_duplicates(),
    on=keys,
    how='outer',
    indicator=True
)

in_cms     = both[both['_merge'] == 'both']
only_in_df  = both[both['_merge'] == 'right_only']   # map nodes with no CMS row
only_in_cms = both[both['_merge'] == 'left_only']  # cms rows without correcposing map enteirs

both['_merge'].value_counts()

_merge
both          63
left_only     46
right_only    40
Name: count, dtype: int64

## Prep for a major bulk change

In [55]:
good_cols = keys + ['node_title','new_description','source','cross_link','data_source_link','inter_hub_link_name','inter_hub_link_url']
cms_short = cms_data[ good_cols ]
cms_short.head()

,map_id,item_id,sublayer_id,kind,node_title,new_description,source,cross_link,data_source_link,inter_hub_link_name,inter_hub_link_url
0,aba702c5420e4a36ac645f14a00ba8f1,d4f639dd817a435fb4bb68fc6102b9c9,NaN,group,Land Cover,Where and how a community grows shapes its inf...,<NA>,<NA>,<NA>,<NA>,<NA>
1,aba702c5420e4a36ac645f14a00ba8f1,c8cb5376347542f597437060984d4069,NaN,layer,Developed Land Change,Land use change aggregated to 1 mile hex bins ...,"U.S. Geological Survey (USGS), National Land C...",<NA>,https://www.mrlc.gov/,<NA>,<NA>
2,aba702c5420e4a36ac645f14a00ba8f1,c8cb5376347542f597437060984d4069,NaN,layer,Natural Cover Change,How much of each 1-mile area changed to or fro...,"U.S. Geological Survey (USGS), National Land C...",<NA>,https://www.mrlc.gov/,<NA>,<NA>
3,aba702c5420e4a36ac645f14a00ba8f1,d54eeaddbc3e442eae779aacc06fdf7c,NaN,layer,Land Cover Change 2004 – 2024 60 meter resolution,Land cover from 2004 and 2024 were first aggre...,"U.S. Geological Survey (USGS), National Land C...",<NA>,https://www.mrlc.gov/,Model Variable Explorer,https://thrive-data-portal.vercel.app/resource...
4,aba702c5420e4a36ac645f14a00ba8f1,13c415b776c0463e863f89361ed688da,NaN,layer,Future Land Cover: 2050,Land cover modeled to represent 2050.,"Clark Labs, Future Land Cover 2050 (Esri Livin...",<NA>,https://livingatlas.arcgis.com/landcover-2050/,<NA>,<NA>


In [56]:
# This is pretty good CMS underlying data
df_merged = df.drop(columns = ['layer_type'] ).merge( cms_short , on=keys, how='left' )


df_merged = df_merged.rename(columns = {'group':'group_path'})

to_add = ['node_type','summary','description','layer_type']
for c in to_add:
    df_merged[c] = None



df_merged.sample(5)

,map_slug,map_title,map_id,group_path,layer_title,kind,item_id,sublayer_id,url,node_title,new_description,source,cross_link,data_source_link,inter_hub_link_name,inter_hub_link_url,node_type,summary,description,layer_type
88,regional-activity,Thrive Projects Activity Map Survey,79e19dd5e63645beb992122c502b69e5,Planning Commissions + Districts,Local Development Districts,layer,931c4e65ca6f40128d1b0b6e67c69dce,NaN,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None
80,transportation-infrastructure,Transportation + Infrastructure,25c75ac490b94b85b02aa4fd2f341fbb,Freight Infrastructure,Rail Network,sublayer,5fcc5ea33daf4d02bd02bbb14a09ef67,120,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None
37,transportation-infrastructure,Transportation + Infrastructure,25c75ac490b94b85b02aa4fd2f341fbb,Cross Sector Data,NaN,group,b29addef77c445cb993be700c67b628e,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None
8,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Cover,Impervious Surface 2005,layer,3c3d68e15b58450b98a16714164d9cbe,NaN,https://tiledimageservices3.arcgis.com/xpR2E2r...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None
97,regional-activity,Thrive Projects Activity Map Survey,79e19dd5e63645beb992122c502b69e5,Thrive Initiatives,NaN,group,e00604c041c147ba8bf1c126b7a3451d,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None


In [ ]:
# CMS Alternative?
df_simple = df_merged.copy()

cols = ['map_title','group_path','kind','layer_title','node_title','new_description','source','cross_link','data_source_link','inter_hub_link_name','inter_hub_link_url' ]
df_simple = df_simple[ cols ].copy()

df_simple.loc[ df_simple['kind'] == 'group' , 'layer_title'] = df_simple.loc[ df_simple['kind'] == 'group' ]['group_path']
df_simple = df_simple.drop( columns = ['kind','group_path'] )

print( list(df_simple['map_title'].unique()) )

df_simple = df_simple.rename( columns={'node_title': "layer_title_display", "layer_title":'layer_title_inmap'})

df_simple[ df_simple['map_title'] == 'Transportation + Infrastructure' ]

['Natural Treasures', 'Transportation + Infrastructure', 'Thrive Projects Activity Map Survey']


,map_title,layer_title_inmap,layer_title_display,new_description,source,cross_link,data_source_link,inter_hub_link_name,inter_hub_link_url
37,Transportation + Infrastructure,Cross Sector Data,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
38,Transportation + Infrastructure,Parcel Boundaries,Regrid USA Nationwide Parcel Boundaries,Regrid Nationwide Parcel Boundaries for the Un...,Regrid Nationwide Parcel Boundaries for the Un...,<NA>,<NA>,<NA>,<NA>
39,Transportation + Infrastructure,Protected Lands,Protected Lands,"Protected lands for the 16-county Chattanooga,...","USGS PAD_US, National Conservation Easement, a...",<NA>,<NA>,<NA>,<NA>
40,Transportation + Infrastructure,Conservation Priority Areas,Conservation Priority Areas,Conservation priority areas are especially val...,UTC,<NA>,<NA>,<NA>,<NA>
41,Transportation + Infrastructure,Impervious Surface 2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
42,Transportation + Infrastructure,Impervious Surface 2005–2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
43,Transportation + Infrastructure,Flood Hazard Areas,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
44,Transportation + Infrastructure,Population Change (2000-2020),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
45,Transportation + Infrastructure,Social Vulnerability Index,Social Vulnerability Index,place holder,Center for Disease Control and Prevention (CDC...,<NA>,<NA>,<NA>,<NA>
46,Transportation + Infrastructure,Stream health,Stream health ratings (2025),"As part of the Clean Water Act, waterbodies ar...",<NA>,<NA>,<NA>,<NA>,<NA>


In [66]:
df_simple.to_excel(r"/mnt/c/Users/cansu/Dropbox/studio2263/Thrive/data/cms-backup/cms_simple.xlsx", index=False )